# Análisis de costos logísticos

Análisis realizado con la finalidad de poder distribuir los costos logisticos asociados al centro de distribución a cada centro de costo de venta/comercial de la organizacion. 

## Librerias

In [1]:
#importamos las librerias que necesitaremos mas adelante

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import psycopg2 
from datetime import datetime 
import time 
import math 
ii = time.time() 
pd.set_option('display.max_rows',None) 
pd.set_option('display.max_columns', None) 

## Se solicitan algunos datos al usuario

In [ ]:
tiempo_muerto = input('ingresar tiempo muerto en minutos: ')
fecha1 = input('ingresar fecha de inicio del periodo: ej: 2022-01-01 ')
fecha2 = input('ingresar fecha finalizacion del periodo: ej: 2022-01-31 ')
nombre_excel = input('ingrese el nombre del archivo excel a generar: ej: C:/Users/fagon/.anaconda/Navigator/mayo2 ')

ingresar tiempo muerto en minutos:  20
ingresar fecha de inicio del periodo: ej: 2022-01-01  2023-06-01
ingresar fecha finalizacion del periodo: ej: 2022-01-31  2023-06-30
ingrese el nombre del archivo excel a generar: ej: C:/Users/fagon/.anaconda/Navigator/mayo2  2023-06


In [4]:
fecha2

'2023-06-30'

In [5]:
nombre_json = nombre_excel + '.json'
nombre_excel = nombre_excel + '.xlsx'
fecha1 = "'" + fecha1 + "'"
fecha2 = fecha2.split('-')
fecha2 = str(pd.Timestamp(int(fecha2[0]),int(fecha2[1]),int(fecha2[2])) + pd.Timedelta(days = 1))[0:10]
fecha2 = "'" + fecha2 + "'"
tiempo_muerto = int(tiempo_muerto)/60

In [6]:
fecha2

"'2023-07-01'"

## Funciones

En esta sección dejaremos todas las funciones que ocuparemos en el codigo mas adelante.

In [7]:
def hora(fecha): # para obtener la hora como un numero entero de la fecha_ingreso como timestamp
  return fecha.hour

def Fecha(fecha): # se obtiene la fecha en formato año-mes-dia 
    return datetime(fecha.year,fecha.month,fecha.day)

def horario(hora): ## para diferenciar entre jornada de dia, noche y horas extras 
    if (hora < 8):
        return "noche"
    elif (hora< 18):
        return "dia"
    elif (hora < 20 ):
        return "extra"
    else: 
        return "noche"

def mes(fecha):
    return fecha.month

def año(fecha):
    return fecha.year

def dia(fecha):
    return fecha.day

def timedelta_hrs(fecha):
  return fecha.total_seconds()/3600 #podria ser horas 

def ajuste(tiempo):
    if tiempo > tiempo_muerto:
        return 2/3*tiempo_muerto
    else: 
        return tiempo 

def super_proceso(actividad):
    if 'DEVOL' in actividad:
        return 'Devolución'
    elif 'PICKING' in actividad:
        return 'Picking'
    elif 'PACKING' in actividad:
        return 'Packing'
    elif 'CAJA' in actividad:
        return 'Cajas'
    elif 'ETIQUETADO' in actividad:
        return 'Etiquetado'
    elif 'CONTROL' in actividad:
        return 'Control'
    elif 'INVENTARIO' in actividad:
        return 'Inventario'
    elif 'PALLET' in actividad:
        return 'Pallets'
    elif 'IMPORTACION' in actividad:
        return 'Importacion'
    elif 'AUDITORIA' in actividad:
        return 'Auditoria'
    else:
        return actividad # quizas mejor "Otros"

In [8]:
def qty(df):
    df = (df.sort_values(["fecha"] ,axis=0)
            .groupby([
                      "fecha",
                      "fecha_siguiente",
                      "intervalo",
                      "usuario",
                      "grupo_producto",
                      "actividad",
                      "canal",
                      "colaborador",
                      "area",
                      "id_pedido",
                      "id_inventario",
                      "nombre_proceso"
                     ]).agg(qty = ('cantidad','sum')).reset_index()
       )
    return df    

In [9]:
def ajuste_diferencia(df):
    df_aux = df.groupby(['usuario', 'fecha', 'fecha_siguiente']).agg(Q = ('qty','sum'))
    df = df.merge(df_aux, how = 'left', on = ['usuario','fecha','fecha_siguiente']).assign(diferencia = lambda x: x.qty*x.intervalo/x.Q) 
    df = df.drop(['Q','intervalo'], axis = 1)
    return df

def get_canal(x):
    if x['nombre_proceso'] == 'none_proceso':
        return 'Curauma'
    elif (x['canal'] == 'Tiendas Amphora') & ('off-line' in x['super_proceso'].lower()):
        return 'Curauma'
    elif (x['canal'] == 'Tiendas Amphora') & ('inventario' in x['super_proceso'].lower()):
        return 'Curauma'
    elif (x['canal'] == 'Tiendas Scalpers') & ('inventario' in x['super_proceso'].lower()):
        return 'Curauma'
    else:
        return x['canal']


def apply(df):
    df["hora"] = df.fecha.apply(hora)
    df["Fecha"] = df.fecha.apply(Fecha)
    df["horario"] = df.hora.apply(horario)
    df["año"] = df.fecha.apply(año)
    df["mes"] = df.fecha.apply(mes)
    df["dia"] = df.fecha.apply(dia)
    df["super_proceso"] = df.actividad.apply(super_proceso)
    # df['canal'] = df.apply(lambda x: x['canal'] if x['nombre_proceso'] != 'none_proceso' else 'Curauma', axis = 1)
    df['canal'] = df.apply(lambda x: get_canal(x), axis = 1)
    return df

In [10]:
def get_suc(proceso): 
    for s in ss:
        if s in proceso.lower().strip():
            return dicc_suc_amp[s]
    if 'RIPLEY' in proceso:
        return 'RIPLEY ' + proceso.split('RIPLEY')[1][0:-10].strip()
    elif 'PUNTO FAL.' in proceso:
        return proceso.split('PUNTO')[1][0:-11].strip()
    elif 'san pedro' in proceso.lower():
        return 'Outlet San Pedro'
    elif 'trebol' in proceso.lower():
        return 'El Trebol'
    elif 'temuco' in proceso.lower():
        return 'Temuco'
    else: 
        return None
        


## Acceso a los Datos

Aqui obtenemos todos los registros de actividades realizadas en el CD en el perido en estudio y lo guardamos en un DataFrame de la libreria pandas.

In [11]:
conexion = psycopg2.connect(host="186.10.254.134", database="packing", user="wmsconsulta", password="wms*consulta", port=5477)

cursor = conexion.cursor()


# consulta =  ("select hp.fecha,"
#             + "hp.fecha_siguiente,"
#         + " hp.tiempo as intervalo,"
#         + " hp.usuario,"
#         + " hp.cantidad,"
#         + " hp.grupo as grupo_producto,"
#         + " hp.actividad,"
#         + " hp.canal,hp.nombre as colaborador, hp.area, hp.id_pedido, hp.id_inventario,hp.nombre_proceso"
#         + " from public.hechos_proceso hp"
#         + " where hp.fecha between" + fecha1 + "and" + fecha2 )

consulta =  ("select hp.fecha,"
            + "hp.fecha_siguiente,"
        + " hp.tiempo as intervalo,"
        + " hp.usuario,"
        + " hp.cantidad,"
        + " hp.grupo as grupo_producto,"
        + " hp.actividad,"
        + " hp.canal,"
        + " hp.nombre as colaborador,"
        + " hp.area,"
        + " hp.id_pedido,"
        + " hp.id_inventario,"
        + " hp.nombre_proceso,"
        + " p.nombre "

        + " from public.hechos_proceso hp left join public.pedido p on p.id = hp.id_pedido "

        + " where hp.fecha between " + fecha1 + " and " + fecha2 )

In [12]:
i = time.time()
cursor.execute(consulta)
f = time.time()
(f-i)/60

0.6107700824737549

In [13]:
data = cursor.fetchall()
name_col = [cursor.description[j][0] for j in range(len(cursor.description))]

In [14]:
df = pd.DataFrame(data, columns = name_col)   #.rename(columns = dict_col )

In [15]:
df.shape

(785376, 14)

In [16]:
df.head()

,fecha,fecha_siguiente,intervalo,usuario,cantidad,grupo_producto,actividad,canal,colaborador,area,id_pedido,id_inventario,nombre_proceso,nombre
0,2023-06-01 08:32:18,2023-06-01 08:32:24,0 days 00:00:06,17559569,1.0,None,CAJA_TRANSITO,Ventas online Multicanal e-commerce Retail,PABLO BRAVO,None,48160808.0,NaN,MC SCL Cyber Multi 30.05.2023,MC SCL Cyber Multi 30.05.2023
1,2023-06-01 08:32:18,2023-06-01 08:32:24,0 days 00:00:06,17559569,1.0,None,CAJA_TRANSITO,Ventas online Multicanal e-commerce Retail,PABLO BRAVO,None,48160808.0,NaN,MC SCL Cyber Multi 30.05.2023,MC SCL Cyber Multi 30.05.2023
2,2023-06-01 08:32:24,2023-06-01 08:32:25,0 days 00:00:01,17559569,1.0,None,CAJA_TRANSITO,Ventas online Multicanal e-commerce Retail,PABLO BRAVO,None,48160808.0,NaN,MC SCL Cyber Multi 30.05.2023,MC SCL Cyber Multi 30.05.2023
3,2023-06-01 08:32:24,2023-06-01 08:32:25,0 days 00:00:01,17559569,1.0,None,CAJA_TRANSITO,Ventas online Multicanal e-commerce Retail,PABLO BRAVO,None,48160808.0,NaN,MC SCL Cyber Multi 30.05.2023,MC SCL Cyber Multi 30.05.2023
4,2023-06-01 08:32:26,2023-06-01 08:32:27,0 days 00:00:01,17559569,1.0,None,CAJA_TRANSITO,Ventas online Multicanal e-commerce Retail,PABLO BRAVO,None,48160808.0,NaN,MC SCL Cyber Multi 30.05.2023,MC SCL Cyber Multi 30.05.2023


In [17]:
df.shape

(785376, 14)

## Procesamiento de los datos

In [18]:
df["id_inventario"] = df["id_inventario"].replace([np.nan],["nan_inventario"])
df["area"] = df["area"].replace([None],["none_area"])
df["id_pedido"] = df["id_pedido"].replace([np.nan],["none_pedido"])
df["nombre_proceso"] = df["nombre_proceso"].replace([None],["none_proceso"])
df["grupo_producto"] = df["grupo_producto"].replace([None],["none_grupo_prod"])
df["fecha_siguiente"] = df["fecha_siguiente"].replace([None],[pd.Timestamp(fecha2)])
df["intervalo"] = df["intervalo"].replace([None],[pd.Timedelta(minutes = 2*tiempo_muerto/3)])

In [19]:
df = qty(df)
df.shape

(316867, 13)

In [20]:
df["intervalo"] = df.intervalo.apply(timedelta_hrs)
df["intervalo"] = df.intervalo.apply(ajuste)

In [21]:
# def ajuste_diferencia(df):
#     df_aux = df.groupby(['usuario', 'fecha', 'fecha_siguiente']).agg(Q = ('qty','sum'))
#     df = df.merge(df_aux, how = 'left', on = ['usuario','fecha','fecha_siguiente']).assign(diferencia = lambda x: x.qty*x.intervalo/x.Q) 
#     df = df.drop(['Q','intervalo'], axis = 1)
#     return df

In [22]:
i = time.time() 
df = ajuste_diferencia(df) #multicanal)
f = time.time()
(f-i)/60

0.01364524761835734

In [23]:
df.head()

,fecha,fecha_siguiente,usuario,grupo_producto,actividad,canal,colaborador,area,id_pedido,id_inventario,nombre_proceso,qty,diferencia
0,2023-06-01 08:29:25,2023-06-01 08:30:50,09684690,none_grupo_prod,CAJA_SELLADO,Ventas online Multicanal e-commerce Inter,LEOPOLDO VATTUONE,none_area,48167531.0,nan_inventario,MC Propio Cyber Multi 31.05.2023,2.0,0.023611
1,2023-06-01 08:29:44,2023-06-01 08:30:49,12225888,none_grupo_prod,CAJA_SELLADO,Ventas online Multicanal e-commerce Inter,SALVADOR CUBILLOS,none_area,48167531.0,nan_inventario,MC Propio Cyber Multi 31.05.2023,2.0,0.018056
2,2023-06-01 08:30:49,2023-06-01 08:33:16,12225888,none_grupo_prod,CAJA_SELLADO,Ventas online Multicanal e-commerce Inter,SALVADOR CUBILLOS,none_area,48167531.0,nan_inventario,MC Propio Cyber Multi 31.05.2023,2.0,0.040833
3,2023-06-01 08:30:50,2023-06-01 08:32:55,09684690,none_grupo_prod,CAJA_SELLADO,Ventas online Multicanal e-commerce Inter,LEOPOLDO VATTUONE,none_area,48167531.0,nan_inventario,MC Propio Cyber Multi 31.05.2023,2.0,0.034722
4,2023-06-01 08:31:09,2023-06-01 08:42:02,19327983,none_grupo_prod,CAJA_ARMADO,WEB Scalpers,CAMILA SALDIVIA,none_area,48172245.0,nan_inventario,SCL Multi 31.05.2023,2.0,0.181389


In [24]:
df = apply(df)

In [25]:
aux = df 

In [26]:
df = (df.groupby(['hora',
                  'horario',
                  'Fecha',
                  'dia',
                  'mes',
                  'año',
                  'area',
                  'id_pedido',
                  'usuario',
                  'colaborador',
                  'canal',
                  'grupo_producto',
                  'actividad',
                  'super_proceso',
                  'nombre_proceso'])
            .agg(tiempo_proceso2 = ("diferencia","sum"),
                                qty2 = ("qty","sum"))).reset_index()

In [27]:
np.sort(df.canal.unique())

array(['Artículos genéricos', 'Consumo Interno', 'Curauma',
       'Exp. ARGENTINA', 'Exp. PERU',
       'Falabella Concesión Marro Sintético', 'Falabella Venta Carteras',
       'Falabella Venta Internet', 'Falabella Venta Internet Scalpers',
       'Falabella Venta Scalpers', 'Gerencia General', 'Importaciones CD',
       'MARKETING', 'Market Dafiti Interandina', 'Market Place Falabella',
       'Market Place Falabella Scalpers', 'Market Place Mercado Libre',
       'Market Place Paris', 'Market Place Ripley', 'Mayor', 'Muestras',
       'Otros', 'Ripley Concesión Carteras', 'Ripley Concesión Marro',
       'Tiendas Amphora', 'Tiendas Scalpers', 'Ventas Directas Tiendas',
       'Ventas online Multicanal e-commerce Inter',
       'Ventas online Multicanal e-commerce Retail', 'WEB Scalpers',
       'Web Amphora', 'Web Ziol'], dtype=object)

## Caso especial MULTICANAL

Distribuimos los registros de "Ventas online Multicanal e-commerce" en todos los canales que lo componen de forma porcentual según la cantidad de productos.

### Ventas online Multicanal e-commerce Inter (MCHEC)

Primero obtenemos los porcentajes de cada canal que compone a "Ventas online Multicanal e-commerce Inter"

In [28]:
df.shape

(23153, 17)

In [29]:
conexion2 = psycopg2.connect(host="186.10.254.134", database="packing", user="wmsconsulta", password="wms*consulta", port=5477)
cursor2 = conexion2.cursor()
consulta2 = ("select case when sd.centrocosto = '702' then 'Web Ziol' when sd.origenreferencia = 'magento' then 'Web Amphora' when sd.origenreferencia = 'ziol' then 'Web Ziol' when sd.origenreferencia = 'ziol magento' then 'Web Ziol'  when sd.origenreferencia = 'dafiti' then 'Market Dafiti Scalpers'when sd.origenreferencia = 'fcom' then 'Market Falabella Amphora' when sd.origenreferencia = 'mercadolibre' then 'Market Mercado Libre' when sd.origenreferencia = 'paris' then 'Market Paris' when sd.origenreferencia = 'ripley' then 'Market Ripley' when sd.origenreferencia = 'scalpers' then 'Web Scalpers'"
           + "else sd.origenreferencia end as canal_origen, count(sd.referencia) as orders, sum(sd.cantproductos) as qty "
           + "from sap_documento sd"
           + " left join tiendaprioridad tp on sd.idsapdoc = tp.idsapdoc join pedido on tp.pedido_id = pedido.id"
           + " where pedido.canal_codigo = 'MCHEC' and sd.origenreferencia is not null and pedido.fecha between "
           + fecha1 + " and " + fecha2 
           + "group by 1")

cursor2.execute(consulta2)
data2 = cursor2.fetchall()
name_col2 = [cursor2.description[j][0] for j in range(len(cursor2.description))]
df2 = (pd.DataFrame(data2, columns = name_col2)
         .assign(porc_qty = lambda df: df.qty/df.qty.sum())
         .drop(["orders","qty"], axis = 1))

In [30]:
df2.sort_values("porc_qty", ascending = False)

,canal_origen,porc_qty
4,Web Amphora,0.808195
1,Market Falabella Amphora,0.108758
5,Web Ziol,0.043036
3,Market Ripley,0.020487
0,Market Dafiti Scalpers,0.010587
2,Market Paris,0.008937


Separamos el conjunto de datos en los registros del Multicanal y los que no.

In [31]:
MC = df[df.canal == "Ventas online Multicanal e-commerce Inter"].drop( ["canal"] ,axis =1)
NMC = df[df.canal != "Ventas online Multicanal e-commerce Inter"].rename(columns = {"tiempo_proceso2" : "tiempo_proceso", "qty2" : "qty"})

In [32]:
MC.shape

(5528, 16)

In [33]:
df.head()

,hora,horario,Fecha,dia,mes,año,area,id_pedido,usuario,colaborador,canal,grupo_producto,actividad,super_proceso,nombre_proceso,tiempo_proceso2,qty2
0,8,dia,2023-06-01,1,6,2023,none_area,48159840.0,17777244,PABLO TRONCOSO,Ventas online Multicanal e-commerce Retail,none_grupo_prod,CAJA_DESPACHADO,Cajas,MC SCL Cyber Mono 30.05.2023,0.122222,18.0
1,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,09684690,LEOPOLDO VATTUONE,Ventas online Multicanal e-commerce Retail,none_grupo_prod,CAJA_SELLADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.312222,6.0
2,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,12225888,SALVADOR CUBILLOS,Ventas online Multicanal e-commerce Retail,none_grupo_prod,CAJA_SELLADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.385556,6.0
3,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,13193027,JUAN CORTES,Ventas online Multicanal e-commerce Retail,none_grupo_prod,CAJA_SELLADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.312778,6.0
4,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,17043407,DANIEL GARATE,Ventas online Multicanal e-commerce Retail,none_grupo_prod,CAJA_RECEPCIONADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.277361,54.0


Generamos los nuevos registros para cada canal de forma proporcional.

In [34]:
MC2 = (MC.merge(df2, how = 'cross')
         .rename(columns = {"canal_origen" : "canal"})
         .assign(tiempo_proceso = lambda x: x.tiempo_proceso2*x.porc_qty,
                            qty = lambda x: x.qty2*x.porc_qty)
         .drop(["tiempo_proceso2","qty2","porc_qty"],axis = 1)).reset_index()

Volvemos a unir toda la data.

In [35]:
df = NMC.merge(MC2, how = 'outer')

In [36]:
df.shape

(50793, 18)

### Ventas online Multicanal e-commerce Retail (MCHES)

In [37]:
conexion6 = psycopg2.connect(host="186.10.254.134", database="packing", user="wmsconsulta", password="wms*consulta", port=5477)
cursor6 = conexion2.cursor()
consulta6 = ("select case when sd.centrocosto = '702' then 'Web Ziol' when sd.origenreferencia = 'magento' then 'Web Amphora' when sd.origenreferencia = 'ziol' then 'Web Ziol' when sd.origenreferencia = 'ziol magento' then 'Web Ziol'  when sd.origenreferencia = 'dafiti' then 'Market Dafiti Scalpers'when sd.origenreferencia = 'fcom' then 'Market Falabella Amphora' when sd.origenreferencia = 'mercadolibre' then 'Market Mercado Libre' when sd.origenreferencia = 'paris' then 'Market Paris' when sd.origenreferencia = 'ripley' then 'Market Ripley' when sd.origenreferencia = 'scalpers' then 'Web Scalpers'"
           + "else sd.origenreferencia end as canal_origen, count(sd.referencia) as orders, sum(sd.cantproductos) as qty "
           + "from sap_documento sd"
           + " left join tiendaprioridad tp on sd.idsapdoc = tp.idsapdoc join pedido on tp.pedido_id = pedido.id"
           + " where pedido.canal_codigo = 'MCHES' and sd.origenreferencia is not null and pedido.fecha between "
           + fecha1 + " and " + fecha2 
           + "group by 1")

cursor6.execute(consulta6)
data6 = cursor6.fetchall()
name_col6 = [cursor6.description[j][0] for j in range(len(cursor6.description))]
df6 = (pd.DataFrame(data6, columns = name_col6)
         .assign(porc_qty = lambda df: df.qty/df.qty.sum())
         .drop(["orders","qty"], axis = 1))

In [38]:
df6.sort_values("porc_qty", ascending = False)

,canal_origen,porc_qty
2,Web Scalpers,0.923913
1,Market Falabella Amphora,0.051630
0,Market Dafiti Scalpers,0.024457


In [39]:
MCS = df[df.canal == "Ventas online Multicanal e-commerce Retail"].drop( ["canal"] ,axis =1)
NMCS = df[df.canal != "Ventas online Multicanal e-commerce Retail"].rename(columns = {"tiempo_proceso2" : "tiempo_proceso", "qty2" : "qty"})

In [40]:
MCS.head()

,hora,horario,Fecha,dia,mes,año,area,id_pedido,usuario,colaborador,grupo_producto,actividad,super_proceso,nombre_proceso,tiempo_proceso,qty,index
0,8,dia,2023-06-01,1,6,2023,none_area,48159840.0,17777244,PABLO TRONCOSO,none_grupo_prod,CAJA_DESPACHADO,Cajas,MC SCL Cyber Mono 30.05.2023,0.122222,18.0,NaN
1,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,09684690,LEOPOLDO VATTUONE,none_grupo_prod,CAJA_SELLADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.312222,6.0,NaN
2,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,12225888,SALVADOR CUBILLOS,none_grupo_prod,CAJA_SELLADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.385556,6.0,NaN
3,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,13193027,JUAN CORTES,none_grupo_prod,CAJA_SELLADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.312778,6.0,NaN
4,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,17043407,DANIEL GARATE,none_grupo_prod,CAJA_RECEPCIONADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.277361,54.0,NaN


In [41]:
MCS2 = (MCS.merge(df6, how = 'cross')
         .rename(columns = {"canal_origen" : "canal"})
         .assign(tiempo_proceso = lambda x: x.tiempo_proceso*x.porc_qty,
                            qty = lambda x: x.qty*x.porc_qty)
         .drop(["porc_qty"],axis = 1)).reset_index()

In [42]:
df = NMCS.merge(MCS2, how = 'outer')

In [43]:
df.shape

(51961, 19)

## Unificación de canales mal nombrados

In [44]:
def renombrar_canal(canal):
    if canal in ["Market Place Falabella","Market Falabella Amphora"]:
        return "Market Place Falabella Amphora"
    elif canal == "Market Ripley":
        return "Market Place Ripley"
    elif canal == "Market Paris":
        return "Market Place Paris"
    elif canal == "linio":
        return "Market Place Linio"
    elif canal == "Market Mercado Libre":
        return "Market Place Mercado Libre"
    # elif canal == "Curauma":
    #     return "(SIN CANAL)"
    else:
        return canal

In [45]:
df["canal"] = df.canal.apply(renombrar_canal)

In [46]:
df = (df.groupby(['hora',
                  'horario',
                  'Fecha',
                  'dia',
                  'mes',
                  'año',
                  'area',
                  'id_pedido',
                  'usuario',
                  'colaborador',
                  'canal',
                  'grupo_producto',
                  'actividad',
                  'super_proceso',
                  'nombre_proceso'])
            .agg(tiempo_proceso = ("tiempo_proceso","sum"),
                                qty = ("qty","sum"))).reset_index()

## Caso especial: Actividades Sin Canal

A continuación se distribuyen las actividades que no tienen un canal asignado a cada uno de los canales presente en el periodo en estudio de forma porcentual y ademas separamos el conjunto de datos en las actividades con y sin canal especificado.

In [47]:
if df[df.canal == "(SIN CANAL)"].shape[0] != 0:
    flag = True
else:
    flag = False

In [48]:
df[df.canal == "(SIN CANAL)"].shape[0]

0

In [49]:
flag

False

In [50]:
if flag:
    SC = df[df.canal == "(SIN CANAL)"].drop(["canal"],axis = 1)
    NSC = df[df.canal != "(SIN CANAL)"]
    pc = (NSC.groupby(["canal"])
             .agg(tiempo = ("tiempo_proceso","sum"),
                      qq = ("qty","sum"))
             .assign(porc_tpo = lambda x: x.tiempo/x.tiempo.sum(),
                      porc_qq = lambda x: x.qq/x.qq.sum())
             .drop(["tiempo","qq"],axis = 1)).reset_index()

Generamos los nuevos registros con cada uno de los canales con su tiempo y cantidad correspondiente.

In [51]:
if flag:
    SC2 = (SC.merge(pc, how = 'cross')
              .assign(tiempo_proceso = lambda x: x.tiempo_proceso*x.porc_tpo,
                                 qty = lambda x: x.qty*x.porc_qq)
              .drop(["porc_tpo","porc_qq"],axis = 1)).reset_index().drop(["index"],axis = 1)

Volvemos a unir todos los datos.

In [52]:
if flag:
    SC2.shape

In [53]:
if flag:
    df = NSC.merge(SC2, how = 'outer')

## Caso especial: Sucursales

Se distribuye los costos de los canales amphora y scalpers en cada una de sus sucursales de manera equitativa segun la cantidad de productos.

### Tiendas Amphora

Obtenemos primero la información con los porcentajes de cantidad vendida:

In [54]:
conexion3 = psycopg2.connect(host="186.10.254.134", database="packing", user="wmsconsulta", password="wms*consulta", port=5477)
cursor3 = conexion3.cursor()
consulta3 = ("select tp.nombrealternativo as sucursal,"
          +        " count(sd.referencia) as orders,"
          +        " sum(sd.cantproductos) as qty "
          + "from sap_documento sd "
          + "left join tiendaprioridad tp on sd.idsapdoc = tp.idsapdoc "
          + " join pedido on tp.pedido_id = pedido.id "
          + " join pedidodetalle detalle on pedido.id = detalle.pedido_id and detalle.tienda_codigo = tp.tienda_codigo "  
          + " join producto on detalle.producto_codigo = producto.codigo "  
          + " join productogrupo on producto.superfamilia = productogrupo.superfamilia "   
          + " where productogrupo.grupo <> 'INSUMO' and tp.fechadespacho is not null and pedido.canal_codigo = 'TIEND' and pedido.fecha between "
          + fecha1 + " and " +  fecha2
          + " group by 1 ")

cursor3.execute(consulta3)
data3 = cursor3.fetchall()
name_col3 = [cursor3.description[j][0] for j in range(len(cursor3.description))]
df3 = (pd.DataFrame(data3, columns = name_col3)
         .assign(porc_qty = lambda df: df.qty/df.qty.sum())
         .drop(["orders","qty"], axis = 1))

In [55]:
df3.sort_values("porc_qty", ascending = False)

,sucursal,porc_qty
10,LOC. LA FABRICA (T79),0.081751
8,LOC. CURAUMA (T74),0.070786
0,EL LLANO (T89),0.061847
6,LOC. COQUIMBO (T82),0.061686
16,LOC. SAN PEDRO DE LA PAZ (T72),0.059522
23,LOC.PLAZA DEL TREBOL (T10),0.059479
9,LOC. EGAñA (T68),0.054080
18,LOC. VIÑA PARK (T70),0.046080
4,LOC. CENTRO CONCEPCION (T64),0.039837
24,LOC.PLAZA VESPUCIO (T34),0.036755


In [56]:
# scalpers.nombre_proceso.unique()

In [57]:
# obtenemos diccionario con nombre sucursal limpio y original.

df3['suc2'] = df3.sucursal.apply(lambda x: str(x).split('(')[0].lstrip('LOC.').strip().lower() if '(' in x else x.lstrip('LOC.').strip().lower() )
ss = df3.suc2.unique().tolist()
dicc_suc_amp = dict(zip(df3.suc2.unique().tolist(),df3.sucursal.unique().tolist()))

In [58]:
amphora = df[df.canal == "Tiendas Amphora"]
df_aux = df[df.canal != "Tiendas Amphora"]

In [59]:
amphora['sucursal'] = amphora.nombre_proceso.apply(lambda x: get_suc(x))

C:\Users\Usuario\AppData\Local\Temp\ipykernel_22600\3455891858.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  amphora['sucursal'] = amphora.nombre_proceso.apply(lambda x: get_suc(x))


In [60]:
amphora_notna = amphora[amphora.sucursal.notna()]
amphora_isna = amphora[amphora.sucursal.isna()].drop(['sucursal'],axis =1)

In [61]:
# amphora_notna.tiempo_proceso.sum()

In [62]:
# amphora_isna.tiempo_proceso.sum() #unique()

In [63]:
# amphora_isna.groupby('nombre_proceso').agg(tiempo = ('tiempo_proceso','sum')).reset_index().sort_values('tiempo', ascending = False) #unique()

In [64]:
# df.tiempo_proceso.sum()

In [65]:
amphora2 =  (amphora_isna.merge(df3, how = 'cross')
         .assign(tiempo_proceso = lambda x: x.tiempo_proceso*x.porc_qty,
                            qty = lambda x: x.qty*x.porc_qty)
         .drop(["porc_qty", "suc2"],axis = 1)).reset_index().drop(["index"], axis = 1)

In [66]:
amphora2.columns

Index(['hora', 'horario', 'Fecha', 'dia', 'mes', 'año', 'area', 'id_pedido',
       'usuario', 'colaborador', 'canal', 'grupo_producto', 'actividad',
       'super_proceso', 'nombre_proceso', 'tiempo_proceso', 'qty', 'sucursal'],
      dtype='object')

In [67]:
amphora3 = amphora2.merge(amphora_notna, how = 'outer')

In [68]:
df = df_aux.merge(amphora3, how = 'outer')

### Tiendas Scalpers

In [69]:
conexion4 = psycopg2.connect(host="186.10.254.134", database="packing", user="wmsconsulta", password="wms*consulta", port=5477)
cursor4 = conexion4.cursor()
consulta4 =  ("select tp.nombrealternativo as sucursal,"
          +        " count(sd.referencia) as orders,"
          +        " sum(sd.cantproductos) as qty "
          + "from sap_documento sd "
          + "left join tiendaprioridad tp on sd.idsapdoc = tp.idsapdoc "
          + " join pedido on tp.pedido_id = pedido.id "
          + " join pedidodetalle detalle on pedido.id = detalle.pedido_id and detalle.tienda_codigo = tp.tienda_codigo "  
          + " join producto on detalle.producto_codigo = producto.codigo "  
          + " join productogrupo on producto.superfamilia = productogrupo.superfamilia "   
          + " where productogrupo.grupo <> 'INSUMO' and tp.fechadespacho is not null and pedido.canal_codigo = 'SCTIE' and pedido.fecha between "
          + fecha1 + " and " +  fecha2
          + " group by 1 ")


cursor4.execute(consulta4)
data4 = cursor4.fetchall()
name_col4 = [cursor4.description[j][0] for j in range(len(cursor4.description))]
df4 = (pd.DataFrame(data4, columns = name_col4)
         .assign(porc_qty = lambda df: df.qty/df.qty.sum())
         .drop(["orders","qty"], axis = 1))

In [70]:
df4.sort_values("porc_qty", ascending = False)

,sucursal,porc_qty
0,SCALPERS ALTO LAS CONDES,0.614935
1,SCALPERS CASA COSTANERA,0.385065


In [71]:
# obtenemos diccionario con nombre sucursal limpio y original.

df4['suc2'] = df4.sucursal.apply(lambda x: str(x).lstrip('SCALPERS').strip().lower()  )
ss = df4.suc2.unique().tolist()
dicc_suc_amp = dict(zip(df4.suc2.unique().tolist(),df4.sucursal.unique().tolist()))

In [72]:
# dicc_suc_amp

In [73]:
df[(df.canal == 'Tiendas Scalpers') ].tiempo_proceso.sum()

74.23185185175926

In [74]:
df[(df.canal == 'Tiendas Scalpers') ][df[(df.canal == 'Tiendas Scalpers') ].nombre_proceso.apply(lambda x : get_suc(x)).isna()].tiempo_proceso.sum() #nombre_proceso.unique() #apply(lambda x : get_suc(x)).isna()

4.423055555555556

In [75]:
scalpers = df[df.canal == "Tiendas Scalpers"].drop(["sucursal"],axis = 1)
df_aux = df[df.canal != "Tiendas Scalpers"]

In [76]:
scalpers['sucursal'] = scalpers.nombre_proceso.apply(lambda x : get_suc(x))

In [77]:
scalpers_notna = scalpers[scalpers.sucursal.notna()]
scalpers_isna = scalpers[scalpers.sucursal.isna()].drop(['sucursal'],axis =1)

In [78]:
# distribuimos el resto en todas las sucursales
scalpers2 =  (scalpers_isna.merge(df4, how = 'cross')
         .assign(tiempo_proceso = lambda x: x.tiempo_proceso*x.porc_qty,
                            qty = lambda x: x.qty*x.porc_qty)
         .drop(["porc_qty"],axis = 1)).reset_index()

In [79]:
# df_aux.columns

In [80]:
scalpers3 =  scalpers2.merge(scalpers_notna, how = 'outer')

In [81]:
df_aux = df_aux#.drop(["index"], axis = 1)
scalpers3 = scalpers3.drop(["index"], axis = 1)

In [82]:
df = df_aux.merge(scalpers3, how = 'outer')
df["sucursal"] = df.sucursal.replace(np.nan, "(SIN SUCURSAL)")

In [83]:
# df.tiempo_proceso.sum()

In [84]:
ax = df.groupby(['nombre_proceso']).agg(qty = ('canal', 'nunique'), canal = ('canal', 'unique')).reset_index().sort_values('qty', ascending = False)

In [85]:
np.unique(ax[ax.qty == 2].canal.tolist())

array([], dtype=float64)

In [86]:
# ax.qty.value_counts()

In [87]:
# np.unique(ax[ax.qty == 2].canal.tolist())

In [88]:
df.head()

,hora,horario,Fecha,dia,mes,año,area,id_pedido,usuario,colaborador,canal,grupo_producto,actividad,super_proceso,nombre_proceso,tiempo_proceso,qty,sucursal,suc2
0,8,dia,2023-06-01,1,6,2023,none_area,48159840.0,17777244,PABLO TRONCOSO,Market Dafiti Scalpers,none_grupo_prod,CAJA_DESPACHADO,Cajas,MC SCL Cyber Mono 30.05.2023,0.002989,0.440217,(SIN SUCURSAL),NaN
1,8,dia,2023-06-01,1,6,2023,none_area,48159840.0,17777244,PABLO TRONCOSO,Market Place Falabella Amphora,none_grupo_prod,CAJA_DESPACHADO,Cajas,MC SCL Cyber Mono 30.05.2023,0.006310,0.929348,(SIN SUCURSAL),NaN
2,8,dia,2023-06-01,1,6,2023,none_area,48159840.0,17777244,PABLO TRONCOSO,Web Scalpers,none_grupo_prod,CAJA_DESPACHADO,Cajas,MC SCL Cyber Mono 30.05.2023,0.112923,16.630435,(SIN SUCURSAL),NaN
3,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,09684690,LEOPOLDO VATTUONE,Market Dafiti Scalpers,none_grupo_prod,CAJA_SELLADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.007636,0.146739,(SIN SUCURSAL),NaN
4,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,09684690,LEOPOLDO VATTUONE,Market Place Falabella Amphora,none_grupo_prod,CAJA_SELLADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.016120,0.309783,(SIN SUCURSAL),NaN


In [89]:
df.tiempo_proceso.sum()

3377.936759253611

In [90]:
3299/180

18.32777777777778

In [91]:
df.columns

Index(['hora', 'horario', 'Fecha', 'dia', 'mes', 'año', 'area', 'id_pedido',
       'usuario', 'colaborador', 'canal', 'grupo_producto', 'actividad',
       'super_proceso', 'nombre_proceso', 'tiempo_proceso', 'qty', 'sucursal',
       'suc2'],
      dtype='object')

In [92]:
df['canal'] = df.canal.replace('Otros', 'Curauma').replace('Consumo Interno','Curauma').replace('Gerencia General','Curauma').replace('Muestras','Curauma')

In [93]:
df.groupby('canal').agg(tiempo = ('tiempo_proceso','sum')).reset_index().sort_values('tiempo', ascending = False)

,canal,tiempo
1,Curauma,1020.769769
21,Tiendas Amphora,710.259676
25,Web Amphora,455.235114
23,Ventas Directas Tiendas,293.551088
24,WEB Scalpers,170.111616
19,Ripley Concesión Carteras,106.036227
15,Market Place Mercado Libre,88.483769
4,Falabella Concesión Marro Sintético,87.636157
13,Market Place Falabella Amphora,78.279209
22,Tiendas Scalpers,74.231852


## Ratios finales

In [94]:
tiempo_productivo_cd = df.tiempo_proceso.sum()
df = (df.assign(productividad = lambda df: df.qty/df.tiempo_proceso,
                porc_tiempo = lambda df: df.tiempo_proceso*100/tiempo_productivo_cd,
                tiempo_por_prod = lambda df: 1/df.productividad))

In [95]:
(time.time() - ii)/60

1.683451255162557

In [96]:
df.shape

(121573, 22)

## Exportamos la data

In [97]:
def entero(a):
    if a == int(a):
        return int(a)
    else:
        return int(a) + 1

In [98]:
n = entero(df.shape[0]/1000000)

In [99]:
df["Fecha"] = df.Fecha.astype("str")

In [100]:
writer = pd.ExcelWriter(nombre_excel) #'C:/Users/fagon/.anaconda/Navigator/mayo2.xlsx')
for i in range(n):
    sheetname = 'BBDD' + str(i+1)
    df_sheet = df[(i*1000000 <= df.index) & (df.index < (i+1)*1000000)]
    df_sheet.to_excel(writer,sheet_name= sheetname)
writer.save()
writer.close()

C:\Users\Usuario\AppData\Local\Temp\ipykernel_22600\183240224.py:6: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  writer.save()


In [101]:
df.to_json(nombre_json, orient = 'records')

In [102]:
ff = time.time()
(ff-ii)/60

5.311350079377492

In [103]:
df.head()

,hora,horario,Fecha,dia,mes,año,area,id_pedido,usuario,colaborador,canal,grupo_producto,actividad,super_proceso,nombre_proceso,tiempo_proceso,qty,sucursal,suc2,productividad,porc_tiempo,tiempo_por_prod
0,8,dia,2023-06-01,1,6,2023,none_area,48159840.0,17777244,PABLO TRONCOSO,Market Dafiti Scalpers,none_grupo_prod,CAJA_DESPACHADO,Cajas,MC SCL Cyber Mono 30.05.2023,0.002989,0.440217,(SIN SUCURSAL),NaN,147.272727,0.000088,0.006790
1,8,dia,2023-06-01,1,6,2023,none_area,48159840.0,17777244,PABLO TRONCOSO,Market Place Falabella Amphora,none_grupo_prod,CAJA_DESPACHADO,Cajas,MC SCL Cyber Mono 30.05.2023,0.006310,0.929348,(SIN SUCURSAL),NaN,147.272727,0.000187,0.006790
2,8,dia,2023-06-01,1,6,2023,none_area,48159840.0,17777244,PABLO TRONCOSO,Web Scalpers,none_grupo_prod,CAJA_DESPACHADO,Cajas,MC SCL Cyber Mono 30.05.2023,0.112923,16.630435,(SIN SUCURSAL),NaN,147.272727,0.003343,0.006790
3,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,09684690,LEOPOLDO VATTUONE,Market Dafiti Scalpers,none_grupo_prod,CAJA_SELLADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.007636,0.146739,(SIN SUCURSAL),NaN,19.217082,0.000226,0.052037
4,8,dia,2023-06-01,1,6,2023,none_area,48160808.0,09684690,LEOPOLDO VATTUONE,Market Place Falabella Amphora,none_grupo_prod,CAJA_SELLADO,Cajas,MC SCL Cyber Multi 30.05.2023,0.016120,0.309783,(SIN SUCURSAL),NaN,19.217082,0.000477,0.052037


## Cantidad por usuario

In [104]:
df['qty'] = df.qty.astype(int)

In [105]:
cantidad_por_usuario = (pd.pivot_table(data = df,
                                   margins = True,
                              margins_name = "Total",
                                     index = ["colaborador"],
                                   columns = ["super_proceso"],
                                    values = "qty",
                                   aggfunc = 'sum')
                         .reset_index())

In [106]:
# cantidad_por_usuario.dtypes

In [107]:
qq = cantidad_por_usuario.replace(np.nan,"").sort_values("Total",ascending = False).reset_index().drop(["index"],axis = 1)
qq.columns.rename("i",inplace = True)
qq

i,colaborador,Auditoria,Cajas,Control,DISPLAY_ENVIADO,DISPLAY_RECIBIDO,Devolución,Etiquetado,Importacion,Inventario,PEGADO GUIAS,Packing,Pallets,Picking,Total
0,Total,8460.0,103532.0,212026.0,4399.0,4818.0,27936.0,25954.0,1796096.0,523468.0,13592.0,159335.0,43971.0,347920.0,3271507
1,RICARDO JIMENEZ PEREIRA,,707.0,,,,,,262142.0,59287.0,,688.0,89.0,118490.0,441403
2,HUGO GOLDSWORTHY,,687.0,,291.0,,,,380820.0,10156.0,,676.0,856.0,10643.0,404129
3,ALEJANDRO GONZALEZ,,930.0,,342.0,,,,246892.0,12599.0,,908.0,145.0,55271.0,317087
4,ANDRÉS COLOMA RAMIREZ,,775.0,,238.0,,,,246892.0,14961.0,,766.0,157.0,15041.0,278830
5,MANUEL PINO,,1981.0,,6.0,12.0,,428.0,246892.0,,,,,,249319
6,DANIEL VENEGAS,,,,,,,,,221208.0,,,,,221208
7,JULIO COVARRUBIAS,,923.0,,2.0,10.0,427.0,322.0,206229.0,,,40.0,2.0,220.0,208175
8,MAURICIO LIRA,,1051.0,,,,,184.0,206229.0,,,,,,207464
9,JUAN CORTES,,1470.0,47005.0,125.0,604.0,,4105.0,,,1242.0,19356.0,,3987.0,77894


In [108]:
# z = 10        # cada 10 filas aparace el nombre de las columnas
# m = entero(qq.shape[0]/z) 
# for j in range(entero(m)):
#     ddd = qq[(z*j <= qq.index) & (qq.index < z*(j+1))]
#     display(ddd)

In [109]:
np.sort(df.canal.unique())

array(['Artículos genéricos', 'Curauma', 'Exp. ARGENTINA', 'Exp. PERU',
       'Falabella Concesión Marro Sintético', 'Falabella Venta Carteras',
       'Falabella Venta Internet', 'Falabella Venta Internet Scalpers',
       'Falabella Venta Scalpers', 'Importaciones CD', 'MARKETING',
       'Market Dafiti Interandina', 'Market Dafiti Scalpers',
       'Market Place Falabella Amphora',
       'Market Place Falabella Scalpers', 'Market Place Mercado Libre',
       'Market Place Paris', 'Market Place Ripley', 'Mayor',
       'Ripley Concesión Carteras', 'Ripley Concesión Marro',
       'Tiendas Amphora', 'Tiendas Scalpers', 'Ventas Directas Tiendas',
       'WEB Scalpers', 'Web Amphora', 'Web Scalpers', 'Web Ziol'],
      dtype=object)

In [110]:
# df[(df.canal == 'Otros')].head()

In [111]:
# df.head()